# Neo4j Azure Serverless Connectivity Test

Validates connectivity from a Databricks **serverless** compute environment to a Neo4j cluster
deployed with Azure Private Link (NCC path).
Credentials are read from Databricks Secrets — no hardcoded values.

**About `neo4j.private`:** The hostname `neo4j.private` is the `domain_names` value embedded in the Databricks NCC Private Endpoint rule when `setup-ncc` runs (default; change with `--domain-name`). Databricks creates internal DNS routing automatically so serverless compute resolves `neo4j.private` to the private endpoint IP — no external DNS infrastructure is required. The connection URI is `neo4j://neo4j.private:7687`.

**Why `neo4j://` works here:** The driver issues a ROUTE request and receives a routing table containing each cluster node's public cloudapp.azure.com FQDN. Those FQDNs are not directly reachable from serverless compute (port 7687 is NSG-restricted to the Databricks VNet CIDR), so the driver falls back to the bootstrap router — the ILB, reachable through the Private Link tunnel — for all query execution. `SHOW SERVERS` returns all three cluster members because that reflects Neo4j's internal cluster state. In practice all queries go through the ILB. `bolt://neo4j.private:7687` is equivalent and skips the routing table entirely.

**Tests:**
1. Network (TCP socket to Bolt port) — confirms Private Link DNS resolution and NSG rules
2. Python driver (authenticate + execute query) — confirms Bolt protocol end-to-end
3. Cluster topology (`SHOW SERVERS`) — confirms all cluster nodes are online

---
## Configuration

Connection details are loaded from the Databricks Secrets scope created by:
```
bicep-deploy setup-databricks --scenario peer-databricks-v2025
```

In [ ]:
SCOPE_NAME = "SCOPE_NAME_PLACEHOLDER"
DOMAIN_NAME = "neo4j.private"
BOLT_PORT = 7687

# Serverless compute reaches Neo4j via the NCC Private Link path using the PLS hostname.
# URI is constructed as neo4j://{DOMAIN_NAME}:{BOLT_PORT} = neo4j://neo4j.private:7687
# Do NOT read bolt_uri from secrets: that secret holds the ILB private IP
# (bolt://<lb-ip>:7687), reachable only from VNet-injected classic compute, not serverless.
# neo4j:// (routing mode) works here because server.default_advertised_address is set
# to the ILB frontend IP, so the routing table only contains addresses reachable through
# the Private Link tunnel.
BOLT_URI  = f"neo4j://{DOMAIN_NAME}:{BOLT_PORT}"
USERNAME  = dbutils.secrets.get(SCOPE_NAME, "username")
PASSWORD  = dbutils.secrets.get(SCOPE_NAME, "password")
DATABASE  = dbutils.secrets.get(SCOPE_NAME, "database")

print(f"Config loaded from scope : {SCOPE_NAME}")
print(f"  Domain name : {DOMAIN_NAME}")
print(f"  Bolt URI    : {BOLT_URI}")
print(f"  Database    : {DATABASE}")
print(f"  Username    : {USERNAME}")
print(f"  Password    : ***")

In [ ]:
%pip install neo4j --quiet

---
## Section 1: Network Connectivity (TCP)

**Expected result:** PASS — confirms the NCC Private Link DNS entry resolves and NSG allows Bolt traffic on port 7687.

In [ ]:
import socket
import time

print("=" * 60)
print("TEST: Network Connectivity (TCP)")
print("=" * 60)
print(f"\nTarget: {DOMAIN_NAME}:{BOLT_PORT}")

try:
    start = time.time()
    sock = socket.create_connection((DOMAIN_NAME, BOLT_PORT), timeout=10)
    sock.close()
    elapsed = (time.time() - start) * 1000

    print("\n" + "=" * 60)
    print(">>> CONNECTIVITY VERIFIED <<<")
    print("=" * 60)
    print(f"[PASS] TCP connection established in {elapsed:.1f}ms")
    print(f"\nDetails:")
    print(f"  Host : {DOMAIN_NAME}")
    print(f"  Port : {BOLT_PORT}")
    print(f"  RTT  : {elapsed:.1f}ms")
    print("\nRESULT: Private Link DNS + NSG rules are OPEN")
    print("Status: PASS")

except Exception as e:
    print(f"\n[FAIL] Cannot reach {DOMAIN_NAME}:{BOLT_PORT}")
    print(f"Error : {e}")
    print("\nCheck:")
    print("  - NCC and PE rule are created (setup-ncc completed)")
    print("  - Private endpoint connection is Approved in Azure portal")
    print("  - NSG inbound rule for port 7687 from Databricks serverless CIDR")
    print("  - Neo4j VMSS instances are running")
    print("Status: FAIL")

---
## Section 2: Neo4j Python Driver

**Expected result:** PASS — confirms Bolt authentication and query execution over the Private Link path.

In [ ]:
import time
from neo4j import GraphDatabase

print("=" * 60)
print("TEST: Neo4j Python Driver")
print("=" * 60)
print(f"\nTarget: {BOLT_URI}")

try:
    start = time.time()
    driver = GraphDatabase.driver(BOLT_URI, auth=(USERNAME, PASSWORD))
    driver.verify_connectivity()
    connect_time = (time.time() - start) * 1000

    print("\n" + "=" * 60)
    print(">>> AUTHENTICATION SUCCESSFUL <<<")
    print("=" * 60)
    print(f"[PASS] Driver connected in {connect_time:.1f}ms")

    with driver.session(database=DATABASE) as session:
        q_start = time.time()
        rec = session.run("RETURN 1 AS test").single()
        q_time = (time.time() - q_start) * 1000
        print(f"[PASS] RETURN 1 = {rec['test']} ({q_time:.1f}ms)")

        components = session.run(
            "CALL dbms.components() YIELD name, versions RETURN name, versions"
        )
        for rec in components:
            print(f"[INFO] {rec['name']} {rec['versions']}")

    driver.close()
    total = (time.time() - start) * 1000
    print(f"\nTotal test time: {total:.1f}ms")
    print("RESULT: Python driver WORKING")
    print("Status: PASS")

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("Status: FAIL")

---
## Section 3: Cluster Topology

**Expected result:** 3 ENABLED servers — confirms all cluster nodes are online and visible via the Private Link path.

In [ ]:
from neo4j import GraphDatabase

print("=" * 60)
print("TEST: Cluster Topology (SHOW SERVERS)")
print("=" * 60)

try:
    driver = GraphDatabase.driver(BOLT_URI, auth=(USERNAME, PASSWORD))

    with driver.session(database="system") as session:
        result = session.run("SHOW SERVERS")
        servers = result.data()

    driver.close()

    enabled = [s for s in servers if s.get("state", "").upper() == "ENABLED"]

    print(f"\nFound {len(servers)} server(s), {len(enabled)} enabled:\n")
    for s in servers:
        state  = s.get("state", "unknown")
        name   = s.get("name", s.get("serverId", "unknown"))
        addr   = s.get("address", "unknown")
        marker = "\u2713" if state.upper() == "ENABLED" else "\u2717"
        print(f"  {marker} {name:<36} {state:<10} {addr}")

    print()
    if len(enabled) >= 3:
        print(f"[PASS] All {len(enabled)} nodes ENABLED")
        print("RESULT: Cluster healthy")
        print("Status: PASS")
    elif len(enabled) > 0:
        print(f"[WARN] Only {len(enabled)}/{len(servers)} nodes enabled — cluster may be degraded")
        print("Status: WARN")
    else:
        print("[FAIL] No enabled nodes found")
        print("Status: FAIL")

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("Status: FAIL")